# Unit 09 · Trust checks: SRM, A/A and novelty · Exercise

<a href="https://colab.research.google.com/github/dinbav/digital-experiments-course/blob/main/notebooks/unit-09-trust-checks-exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The new checkout page shows a +2-point conversion lift, p < 0.001. The team
drafts the launch email. One analyst notices something boring: the experiment
was designed as a 50/50 split, but the treatment group has 2,400 fewer users
than control. Is that a rounding detail - or the whole result?

**After this notebook you can:** reproduce a fake win, catch it with an SRM check, measure an A/A false-positive rate, and detect a novelty effect.

**Time:** about 25 seconds to run in Colab. No accounts, no API keys, no files to upload.

## Without code

You can do this exercise without writing Python. Open the
[worked demo](https://colab.research.google.com/github/dinbav/digital-experiments-course/blob/main/solutions/unit-09-trust-checks-demo.ipynb) (its outputs are saved), then answer the
questions in the **Your answer** section at the end of this notebook in
plain words. What you should find:

1. The logged data shows a significant lift (p < 0.05) for a page that does nothing.
2. The SRM check **fails** (p-value far below 0.001): about 10,000 vs 7,600 users.
3. In A/A simulations, about **5%** of runs come out significant.
4. The week-1 lift is much **larger** than the week-4 lift - novelty.

## 1. The question

Same broken checkout test as the demo. Show the fake win, then run the three
checks that would have stopped the launch email.

## 2. Setup

The next cell loads the libraries and fixes the random seed, so everyone gets
the same numbers. Colab already has every library this notebook needs. If an
import ever fails, delete the `#` in front of `%pip` and run the cell again.

In [ ]:
# %pip install -q numpy pandas scipy statsmodels matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf

RANDOM_SEED = 42  # change it and re-run: the numbers move, the lesson should not
np.random.seed(RANDOM_SEED)
plt.rcParams['figure.figsize'] = (7, 4)

## 3. The data

Given. `logs` is what the analysis pipeline sees (true effect: zero, but some
treated users are missing). `weekly` is a separate four-week experiment.

In [ ]:
# 20,000 users, assigned 50/50. 30% use an old browser and convert less (5% vs 15%).
# The new page has NO effect on conversion - but its new tracking script fails on
# old browsers, so 80% of treated old-browser users never reach the logs.
n = 20_000
D = np.random.binomial(1, 0.5, n)
old_browser = np.random.binomial(1, 0.30, n)
converted = np.random.binomial(1, np.where(old_browser == 1, 0.05, 0.15))
logged = np.where((D == 1) & (old_browser == 1), np.random.binomial(1, 0.20, n), 1)
everyone = pd.DataFrame({'D_i': D, 'old_browser': old_browser, 'converted': converted})
logs = everyone[logged == 1]          # what the analysis pipeline actually sees

# Four weeks, 10,000 new users per arm per week. The real, lasting effect is +0.5 points,
# plus a novelty bump of +2.5 points in week 1 that halves every week.
rows = []
for week in range(1, 5):
    lift = 0.005 + 0.025 * 0.5 ** (week - 1)
    for arm in (0, 1):
        rows.append(pd.DataFrame({'week': week, 'D_i': arm,
                                  'converted': np.random.binomial(1, 0.12 + lift * arm, 10_000)}))
weekly = pd.concat(rows, ignore_index=True)

print('Users in the logs:', len(logs), '  Rows in weekly:', len(weekly))

**How the exercise works.** Each `TODO` cell tells you what to compute. Replace
the `None` with your code and run the cell. The `assert` lines below it check
your answer: if one fails, its message tells you what to look at. The data
cells are given - you only write the parts that carry the idea.

## 4. TODO 1 - the naive result

From `logs`, compute `naive_lift` (treatment minus control conversion) and
`p_value` from a two-sample t-test (`stats.ttest_ind(..., equal_var=False)`).

In [ ]:
# TODO: lift and p-value on the logged data
naive_lift = None
p_value = None

assert naive_lift is not None and p_value is not None, 'Replace both Nones'
print(f'Naive lift: {naive_lift * 100:+.1f} points, p = {p_value:.1e}')
assert naive_lift > 0 and p_value < 0.05, 'The broken pipeline should produce a significant fake win'

## 5. TODO 2 - the SRM check

Complete `srm_pvalue(n_control, n_treatment)` for a designed 50/50 split, using
`stats.chisquare`. Then apply it to the logged counts.

In [ ]:
def srm_pvalue(n_control, n_treatment):
    # TODO: chi-square test of the observed counts against a 50/50 split
    return None

counts = logs['D_i'].value_counts().sort_index()
p_srm = srm_pvalue(counts[0], counts[1])
assert p_srm is not None, 'Complete srm_pvalue'
print(f'Logged counts: {counts[0]:,} vs {counts[1]:,}   SRM p-value: {p_srm:.1e}')
assert p_srm < 0.001, 'This split should fail the SRM check'
assert srm_pvalue(10_000, 9_950) > 0.05, 'A near-even split should pass'

## 5b. TODO 3 - the A/A false-positive rate

Simulate 500 A/A experiments: two groups of 3,000 users, both converting at
12%. Count how often a t-test gives p < 0.05. `aa_rate` is that share.

In [ ]:
# TODO: 500 A/A simulations
aa_rate = None

assert aa_rate is not None, 'Replace None'
print(f'A/A false-positive rate: {aa_rate:.1%}')
assert 0.02 < aa_rate < 0.08, 'Should be close to 5% - are both groups really identical?'

## 5c. TODO 4 - the effect over time

From `weekly`, compute the lift (treatment minus control conversion) **for each
week**. Store it as `weekly_lift`, a pandas Series indexed by week. Then plot it.

In [ ]:
# TODO: lift per week, then a line chart of weekly_lift
weekly_lift = None

assert weekly_lift is not None, 'Replace None'
print((weekly_lift * 100).round(2))
assert weekly_lift.loc[1] > weekly_lift.loc[4], 'The week-1 lift should be larger than week 4'

## Your answer

Double-click this cell and write your answers under each question. One or two
sentences each is enough - we grade the reasoning, not the length.

**1.** The SRM check failed. What would you tell the team that wants to "just analyse the modern-browser users"?

**2.** Your platform's A/A tests come out significant 15% of the time. What does that tell you about its A/B results?

**3.** Name one trust check you will run in your own project, and the result that would make you stop. (This feeds Task 09.)

## Hints

<details>
<summary>Open a hint only after you have tried</summary>

- `logs.groupby('D_i')['converted'].mean()` for the two rates; select each arm with `logs.loc[logs.D_i == 1, 'converted']`.
- `stats.chisquare([c, t], f_exp=[total / 2, total / 2]).pvalue`.
- A/A: in a loop, draw two arrays with `np.random.binomial(1, 0.12, 3000)` and test them.
- Per week: `weekly.groupby(['week', 'D_i'])['converted'].mean().unstack()` gives a week-by-arm table.

</details>

## Spoiler: full solution

Looking before you have tried costs you the exercise - the struggle is where
the learning happens.

<details>
<summary>Show the solution</summary>

```python
g = logs.groupby('D_i')['converted'].mean()
naive_lift = g[1] - g[0]
p_value = stats.ttest_ind(logs.loc[logs.D_i == 1, 'converted'],
                          logs.loc[logs.D_i == 0, 'converted'], equal_var=False).pvalue

def srm_pvalue(n_control, n_treatment):
    total = n_control + n_treatment
    return stats.chisquare([n_control, n_treatment], f_exp=[total / 2, total / 2]).pvalue

hits = 0
for _ in range(500):
    a = np.random.binomial(1, 0.12, 3_000)
    b = np.random.binomial(1, 0.12, 3_000)
    hits += stats.ttest_ind(a, b, equal_var=False).pvalue < 0.05
aa_rate = hits / 500

table = weekly.groupby(['week', 'D_i'])['converted'].mean().unstack()
weekly_lift = table[1] - table[0]
weekly_lift.plot(marker='o', title='Weekly lift')
plt.show()
```

</details>

---

**Takeaway:** Check assignment before effect, and time before verdict.

**Next:** [Task 09](https://github.com/dinbav/digital-experiments-course/blob/main/tasks/task-09.md) · [Open the worked demo in Colab](https://colab.research.google.com/github/dinbav/digital-experiments-course/blob/main/solutions/unit-09-trust-checks-demo.ipynb)

**Further reading:** Kohavi, Tang & Xu, *Trustworthy Online Controlled Experiments* (2020), ch. 3 (Twyman's law), ch. 19 (A/A tests), ch. 21 (sample ratio mismatch) and ch. 23 (long-term effects); Fabijan et al., *Diagnosing Sample Ratio Mismatch in Online Controlled Experiments*, KDD 2019.